# [9.4] White-box Evals and Monitors - Solutions

This notebook runs the reference implementation, visible tests, notebook contract, and committed CUDA report checks for the section.


In [1]:
import json
import sys
from pathlib import Path

chapter = "chapter9_alignment_interpretability"
section = "part4_white_box_evals_monitors"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_white_box_evals_monitors.solutions as solutions
import part4_white_box_evals_monitors.tests as tests


## Visible Unit Tests

These are the same tests students call after each exercise.

<details>
<summary>Expected output</summary>

```text
All tests in `test_monitor_dashboard_row_preserves_review_fields` passed!
All tests in `test_binary_auroc_counts_ties_and_validates_inputs` passed!
All tests in `test_monitor_calibration_report_matches_reference` passed!
All tests in `test_missed_failure_report_identifies_white_box_only_catches` passed!
All tests in `test_false_positive_documentation_requires_notes` passed!
All tests in `test_feature_explanation_validation_uses_heldout_accuracy` passed!
All tests in `test_pythia_monitor_helpers_reject_invalid_internal_evidence` passed!
```

</details>

<details>
<summary>Help - reading the solution tests</summary>

The solution tests do more than check happy paths. They also reject non-finite monitor scores, non-binary labels, one-class AUROC inputs, invalid thresholds, zero monitor directions, and multi-token black-box proxy labels.

</details>


In [2]:
tests.test_monitor_dashboard_row_preserves_review_fields(solutions.monitor_dashboard_row)
tests.test_binary_auroc_counts_ties_and_validates_inputs(solutions.binary_auroc)
tests.test_monitor_calibration_report_matches_reference(solutions.monitor_calibration_report)
tests.test_missed_failure_report_identifies_white_box_only_catches(solutions.missed_failure_report)
tests.test_false_positive_documentation_requires_notes(solutions.false_positive_documentation_report)
tests.test_feature_explanation_validation_uses_heldout_accuracy(solutions.feature_explanation_validation_report)
tests.test_pythia_monitor_helpers_reject_invalid_internal_evidence()


All tests in `test_monitor_dashboard_row_preserves_review_fields` passed!
All tests in `test_binary_auroc_counts_ties_and_validates_inputs` passed!
All tests in `test_monitor_calibration_report_matches_reference` passed!
All tests in `test_missed_failure_report_identifies_white_box_only_catches` passed!
All tests in `test_false_positive_documentation_requires_notes` passed!
All tests in `test_feature_explanation_validation_uses_heldout_accuracy` passed!
All tests in `test_pythia_monitor_helpers_reject_invalid_internal_evidence` passed!


## Notebook Contract

The smoke contract keeps the visible dashboard, calibration, missed-failure, false-positive, and explanation-validation checks together.

<details>
<summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

</details>

<details>
<summary>Help - why keep the contract separate?</summary>

The contract is a small deterministic bundle that protects the notebook surface. It is not the CUDA proof; the real-model evidence comes from the committed verification report and the live report generator.

</details>


In [3]:
contract = solutions.run_smoke_test(cpu=True)
tests.test_notebook_contract(solutions.run_smoke_test)
contract


All tests in `test_notebook_contract` passed!


{'dashboard': {'prompt': 'Summarize this harmless note.',
  'model_output': 'A concise summary.',
  'active_features': ('summary', 'benign'),
  'refusal_score': 0.1,
  'hallucination_score': 0.2,
  'cot_faithfulness_score': 0.9},
 'calibration': {'auroc': 1.0, 'calibrated': True},
 'missed_failure': {'caught_failure_indices': (0,),
  'num_caught_failures': 1,
  'catches_black_box_miss': True},
 'false_positive': {'false_positive_indices': (2,),
  'num_false_positives': 1,
  'documented': True},
 'explanation_validation': {'heldout_accuracy': 1.0,
  'explanations_validated': True}}

## Signature Result

The committed report is the CUDA-backed result used by the lesson page. It is narrow evidence for the safe Pythia hidden-state preflight, not a deployment monitor benchmark.

<details>
<summary>Expected output</summary>

```text
All tests in `test_committed_gpu_report_matches_white_box_monitor_contract` passed!
monitor_auroc: 1.0
black_box_proxy_accuracy: 0.875
black_box_missed_failure_count: 3
label_shuffled_monitor_auroc: 0.3875
random_direction_monitor_auroc: 0.0875
generation_used: false
```

</details>

<details>
<summary>Help - interpreting the signature result</summary>

The important claim is the conjunction: held-out white-box calibration succeeds, the black-box proxy misses some failures, label-shuffled and random-direction controls fail, and the report uses no generated completions.

</details>


In [4]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu_result = report["metrics"]["gpu_test"]
tests.test_committed_gpu_report_matches_white_box_monitor_contract(gpu_result)
summary_keys = [
    "model_name",
    "hf_revision",
    "train_prompt_count",
    "heldout_prompt_count",
    "hidden_state_shape",
    "monitor_auroc",
    "white_box_accuracy",
    "black_box_proxy_accuracy",
    "black_box_missed_failure_count",
    "label_shuffled_monitor_auroc",
    "random_direction_monitor_auroc",
    "heldout_explanation_accuracy",
    "peak_vram_gb",
    "generation_used",
]
{k: gpu_result[k] for k in summary_keys}


All tests in `test_committed_gpu_report_matches_white_box_monitor_contract` passed!


{'model_name': 'EleutherAI/pythia-70m-deduped',
 'hf_revision': 'e93a9faa9c77e5d09219f6c868bfc7a1bd65593c',
 'train_prompt_count': 36,
 'heldout_prompt_count': 24,
 'hidden_state_shape': [24, 512],
 'monitor_auroc': 1.0,
 'white_box_accuracy': 1.0,
 'black_box_proxy_accuracy': 0.875,
 'black_box_missed_failure_count': 3,
 'label_shuffled_monitor_auroc': 0.3875,
 'random_direction_monitor_auroc': 0.0875,
 'heldout_explanation_accuracy': 1.0,
 'peak_vram_gb': 0.30931854248046875,
 'generation_used': False}

## Limitations

The real-model path uses safe monitor records, hidden states, and next-token logits only. It does not run generated-completion evaluations and does not prove broad harmful-content monitoring.
